# Analysis

Sentiment, categorization, clustering, and Plotly exploration for discovery records.

Section 1 runs `extract_records.run()` to build `output/processed/challenges.csv` and `expectations.csv` from source notes/worksheets (no terminal step needed).


In [9]:
%pip install -q -r requirements.txt
%pip install -q "nbformat>=4.2.0"


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 0. Import Libraries

In [10]:
from pathlib import Path
import sys

import hdbscan
import numpy as np
import pandas as pd
import plotly.express as px
import umap
from IPython.display import Markdown, display
from sklearn.metrics import silhouette_score

sys.path.insert(0, str(Path(".").resolve()))
from setup.analyze_records import (
    CATEGORY_CONFIG,
    CATEGORY_DESCRIPTIONS,
    realign_by_sentiment,
    run_categorize,
    run_sentiment,
)
from setup.extract_records import SOURCE_MEETING_NOTES, load_prepared_records, run

CHALLENGES_CSV = Path("./output/processed/challenges.csv")
EXPECTATIONS_CSV = Path("./output/processed/expectations.csv")
PROCESSED_DIR = Path("./output/processed")

CHALLENGES_SCORED = PROCESSED_DIR / "challenges_scored.csv"
EXPECTATIONS_SCORED = PROCESSED_DIR / "expectations_scored.csv"
CATEGORIZED_CHALLENGES_CSV = PROCESSED_DIR / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED_DIR / "categorized_expectations.csv"
CATEGORY_SUMMARY_CSV = PROCESSED_DIR / "category_summary.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


## 1. Extract, load & prep records

Runs `run()` to extract from source notes/worksheets into CSVs, then `load_prepared_records()` (focus-group aliases, source tags, short meeting-note merge).


In [11]:
run()  # writes output/processed/challenges.csv and expectations.csv
df_painpoints, df_expectations = load_prepared_records(
    CHALLENGES_CSV, EXPECTATIONS_CSV
)

print(f"Challenges: {len(df_painpoints)} | Expectations: {len(df_expectations)}")
print(df_painpoints["source"].value_counts().to_string())


FileNotFoundError: [Errno 2] No such file or directory: 'output/processed/challenges.csv'

In [ ]:
df_painpoints.sample(10)

## 2. Sentiment Analysis

Uses `Thi144/sentiment-distilbert-7class`, mapped to negative / neutral / positive.


In [ ]:
df_painpoints, df_expectations = run_sentiment(df_painpoints, df_expectations)

print("Pain sentiment:\n", df_painpoints["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", df_expectations["sentiment"].value_counts().to_string())

df_painpoints.to_csv(CHALLENGES_SCORED, index=False, encoding="utf-8-sig")
df_expectations.to_csv(EXPECTATIONS_SCORED, index=False, encoding="utf-8-sig")
print(f"Saved → {CHALLENGES_SCORED.name}, {EXPECTATIONS_SCORED.name}")


## 3. Realign misclassified rows

Move positive “pain” → expectations and negative “expectations” → pain, **meeting_notes only**.


In [ ]:
n_pos = int(
    ((df_painpoints["source"] == SOURCE_MEETING_NOTES) & (df_painpoints["sentiment"] == "positive")).sum()
)
n_neg = int(
    ((df_expectations["source"] == SOURCE_MEETING_NOTES) & (df_expectations["sentiment"] == "negative")).sum()
)

df_painpoints, df_expectations = realign_by_sentiment(
    df_painpoints, df_expectations, only_source=SOURCE_MEETING_NOTES
)

print(f"Moved meeting_notes only: {n_pos} positive→expectations, {n_neg} negative→pain")
print(f"Shapes: pain {len(df_painpoints)}, expectations {len(df_expectations)}")
print(df_painpoints["source"].value_counts().to_string())


## 4. Categorize records

Hybrid: title keywords → body keywords → semantic similarity.


In [ ]:
df, expectations_df, challenge_embeddings, embedder = run_categorize(
    df_painpoints, df_expectations
)

print("Challenge categories:\n", df["Category"].value_counts().to_string())
print("\nExpectation categories:\n", expectations_df["Category"].value_counts().to_string())


## 5. Cluster challenges (UMAP + HDBSCAN)


In [ ]:
def assign_cluster_names(
    frame: pd.DataFrame,
    cluster_col: str = "Cluster",
    category_col: str = "Category",
    output_col: str = "Cluster_Label",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cluster_summary = (
        frame.groupby([cluster_col, category_col]).size().reset_index(name="Count")
    )
    dominant = (
        cluster_summary.sort_values("Count", ascending=False)
        .drop_duplicates(subset=[cluster_col])
        [[cluster_col, category_col]]
        .rename(columns={category_col: output_col})
    )
    return frame.merge(dominant, on=cluster_col, how="left"), cluster_summary


def soft_assign_noise(
    coords: np.ndarray,
    labels: np.ndarray,
    radius_percentile: float = 92,
    radius_slack: float = 1.35,
) -> tuple[np.ndarray, int]:
    """Pull leftover noise into the nearest cluster if it sits inside that cluster's envelope."""
    out = labels.copy()
    clustered = out != -1
    if not clustered.any():
        return out, 0

    centroids, radii = {}, {}
    for cid in sorted(set(out[clustered])):
        pts = coords[out == cid]
        centroids[cid] = pts.mean(axis=0)
        dists = np.linalg.norm(pts - centroids[cid], axis=1)
        radii[cid] = float(np.percentile(dists, radius_percentile) * radius_slack)

    assigned = 0
    for i in np.where(out == -1)[0]:
        best_c, best_d = None, np.inf
        for cid, center in centroids.items():
            d = float(np.linalg.norm(coords[i] - center))
            if d < best_d:
                best_c, best_d = cid, d
        if best_c is not None and best_d <= radii[best_c]:
            out[i] = best_c
            assigned += 1
    return out, assigned


TARGET_CLUSTERS = len(CATEGORY_CONFIG)
# Prefer coverage over ultra-tight cores (prior sweep left ~36% as noise).
NOISE_PENALTY = 1.35
CLUSTER_COUNT_PENALTY = 0.012

reducer = umap.UMAP(
    n_neighbors=15, n_components=12, min_dist=0.0, metric="cosine", random_state=42
)
reduced = reducer.fit_transform(challenge_embeddings)

candidate_params = sorted(
    {
        (mcs, ms, method)
        for mcs in range(5, 14)
        for ms in (1, 2, 3, max(1, mcs // 3))
        for method in ("eom", "leaf")
    }
)

sweep_rows, best_labels, best_params, best_score = [], None, None, -np.inf
n_rows = max(len(df), 1)
for mcs, ms, method in candidate_params:
    labels = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean",
        cluster_selection_method=method,
    ).fit_predict(reduced)

    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    noise = int((labels == -1).sum())
    noise_frac = noise / n_rows
    silhouette = (
        silhouette_score(reduced[mask], labels[mask], metric="euclidean")
        if mask.sum() > 1 and n_clusters > 1
        else -1.0
    )
    # Reject near-all-noise or absurd fragmentation early.
    if noise_frac > 0.45 or n_clusters < 4 or n_clusters > 40:
        combined = -1.0
    else:
        combined = (
            silhouette
            - abs(n_clusters - TARGET_CLUSTERS) * CLUSTER_COUNT_PENALTY
            - noise_frac * NOISE_PENALTY
        )
    sweep_rows.append({
        "min_cluster_size": mcs,
        "min_samples": ms,
        "method": method,
        "clusters": n_clusters,
        "noise": noise,
        "noise_pct": round(noise_frac * 100, 1),
        "silhouette": round(float(silhouette), 4),
        "combined_score": round(float(combined), 4),
    })
    if combined > best_score:
        best_score, best_labels, best_params = combined, labels, (mcs, ms, method)

print("Best HDBSCAN params:", best_params, "score:", round(float(best_score), 4))
display(pd.DataFrame(sweep_rows).sort_values("combined_score", ascending=False).head(10))

raw_labels = best_labels.copy()
raw_noise = int((raw_labels == -1).sum())
best_labels, n_soft = soft_assign_noise(reduced, best_labels)
print(
    f"Soft-assigned {n_soft} noise points into nearest clusters "
    f"(noise {raw_noise} → {int((best_labels == -1).sum())})"
)

df["Cluster_Raw"] = raw_labels
df["Cluster"] = best_labels
df["Assign_Status"] = np.where(
    raw_labels != -1,
    "HDBSCAN cluster",
    np.where(best_labels != -1, "Soft-assigned", "Unclustered"),
)
df, cluster_summary = assign_cluster_names(df)
df.loc[df["Cluster"] == -1, "Cluster_Label"] = "Noise / unclustered"

print("\nAssignment status:")
print(df["Assign_Status"].value_counts().to_string())
print("\nCluster → label counts:")
print(df.groupby(["Cluster", "Cluster_Label"]).size().to_string())
print(
    f"\nUnclustered: {(df['Cluster'] == -1).sum()} / {len(df)} "
    f"({100 * (df['Cluster'] == -1).mean():.1f}%)"
)


### Soft assignment map

2D projection of the same UMAP space used for HDBSCAN. Color by assignment status to see which points were dense cores, which were rescued by soft assignment, and which stayed unclustered.


### Cluster contents

Browse every cluster for dissection, including noise (`Cluster == -1`).


In [ ]:
cols = [
    c
    for c in [
        "Cluster",
        "Cluster_Label",
        "Assign_Status",
        "focus_group",
        CHALLENGE_TEXT_COL,
        "Category",
        "source",
    ]
    if c in df.columns
]
show = df[cols].sort_values(["Cluster", "Assign_Status", "focus_group"]).reset_index(drop=True)

print("Cluster sizes (including unclustered):")
display(
    show.groupby(["Cluster", "Cluster_Label"], as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Cluster")
)

for cid, group in show.groupby("Cluster", sort=True):
    label = group["Cluster_Label"].iloc[0]
    display(Markdown(f"### Cluster {cid} — {label} ({len(group)} records)"))
    display(
        group[[c for c in ["focus_group", CHALLENGE_TEXT_COL, "Category", "source"] if c in group.columns]]
        .reset_index(drop=True)
    )


In [ ]:
# add a new column called cluster_number
df["cluster_number"] = df["Cluster"]

# add a new column called cluster_label
df["cluster_label"] = df["Cluster_Label"]


## 6. Visualizations


In [ ]:
category_counts = (
    df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain point categories", color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_cat.show()

expectation_counts = (
    expectations_df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_exp_cat = px.bar(
    expectation_counts, x="Count", y="Category", orientation="h",
    title="Expectation categories", color="Count", color_continuous_scale="Teal",
)
fig_exp_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_exp_cat.show()


In [ ]:
coords_2d = umap.UMAP(
    n_neighbors=15, n_components=2, min_dist=0.1, metric="cosine", random_state=42
).fit_transform(challenge_embeddings)
plot_df = pd.DataFrame({
    "x": coords_2d[:, 0], "y": coords_2d[:, 1],
    "Category": df["Category"].values,
    "Cluster_Label": df["Cluster_Label"].values,
    "Challenge": df[CHALLENGE_TEXT_COL].values,
    "Focus Group": df["focus_group"].values,
})
fig_map = px.scatter(
    plot_df, x="x", y="y", color="Category",
    hover_data=["Focus Group", "Cluster_Label", "Challenge"],
    title="2D semantic map by category",
)
fig_map.update_layout(height=560, width=1100, template="plotly_white")
fig_map.show()

In [ ]:
ch_c = df["Category"].value_counts()
ex_c = expectations_df["Category"].value_counts()
cats = sorted(set(ch_c.index) | set(ex_c.index), key=lambda c: -(ch_c.get(c, 0) + ex_c.get(c, 0)))
plot_cmp = pd.DataFrame({
    "Category": cats,
    "Challenges": [int(ch_c.get(c, 0)) for c in cats],
    "Expectations": [int(ex_c.get(c, 0)) for c in cats],
}).melt(id_vars="Category", var_name="Type", value_name="Count")
fig_cmp = px.bar(
    plot_cmp, x="Count", y="Category", color="Type", barmode="group", orientation="h",
    title="Challenges vs expectations by theme",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
)
fig_cmp.update_layout(height=520, template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig_cmp.show()

In [ ]:
top_n = 18
top = df.groupby("focus_group").size().sort_values(ascending=False).head(top_n)
exp = expectations_df.groupby("focus_group").size()
plot_top = pd.DataFrame({
    "Focus Group": top.index.tolist(),
    "Challenges": top.values.astype(int),
    "Expectations": [int(exp.get(g, 0)) for g in top.index],
}).melt(id_vars="Focus Group", var_name="Kind", value_name="Count")
fig_top = px.bar(
    plot_top,
    x="Focus Group",
    y="Count",
    color="Kind",
    barmode="group",
    title=f"Top {top_n} focus groups",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
    category_orders={"Focus Group": top.index.tolist()},
)
fig_top.update_layout(
    height=560,
    width=1100,
    template="plotly_white",
    xaxis=dict(tickangle=-40),
    margin=dict(b=140),
)
fig_top.show()


In [ ]:
heatmap_df = pd.crosstab(df["focus_group"], df["Category"])
focus_group_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[focus_group_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Group", y="Category", color="Count"),
    # title="Focus group vs category",
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(height=700, width=1100, xaxis_tickangle=-45, template="plotly_white")
fig_heatmap.show()

## 7. Save categorized outputs


In [ ]:
challenge_cols = [
    c for c in [
        "department", "focus_group", CHALLENGE_TEXT_COL, "processed_text", "source", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
        "Cluster", "Cluster_Label",
    ] if c in df.columns
]
expectation_cols = [
    c for c in [
        "department", "focus_group", EXPECTATION_TEXT_COL, "processed_text", "source", "sentiment",
        "title", "Category", "Category_Method", "Category_Confidence",
    ] if c in expectations_df.columns
]

df[challenge_cols].to_csv(CATEGORIZED_CHALLENGES_CSV, index=False, encoding="utf-8-sig")
expectations_df[expectation_cols].to_csv(
    CATEGORIZED_EXPECTATIONS_CSV, index=False, encoding="utf-8-sig"
)
category_summary = (
    df.groupby("Category", as_index=False).size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=False)
)
category_summary.to_csv(CATEGORY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"Saved {CATEGORIZED_CHALLENGES_CSV} ({len(df)} rows)")
print(f"Saved {CATEGORIZED_EXPECTATIONS_CSV} ({len(expectations_df)} rows)")
print(f"Saved {CATEGORY_SUMMARY_CSV}")
display(category_summary)
